# Notebook 01: Comprehensive Exploratory Data Analysis (EDA)
**GuidedGuard – Explainable AI for Scam-Guided Digital Payment Detection**

--- 
### EDA Phase Objectives:
Perform extensive, non-destructive exploratory data analysis on `PaySim Mobile Money` (`data/raw/paysim_transactions.csv`) and `Bank Account Fraud (BAF)` (`data/raw/baf_base_dataset.csv`) to analyze data quality, feature distributions, target class imbalance, outliers, correlations, and fraud patterns.

> **Strict Boundary Guardrails:**
> - ❌ No data cleaning performed.
> - ❌ No preprocessing or scaling applied.
> - ❌ No new feature engineering implemented.
> - ❌ No machine learning models trained.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Add project root to sys.path
sys.path.append(str(Path.cwd().parent))
import config
from utils.data_loader import load_dataset, validate_dataset_schema, get_dataset_summary

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (10, 5)
print("EDA Environment initialized successfully.")

# SECTION 1: DATASET OVERVIEW
Structural analysis of PaySim and Bank Account Fraud (BAF) datasets.

In [ ]:
# 1. Load Raw Datasets
paysim_path = config.RAW_DATA_DIR / "paysim_transactions.csv"
baf_path = config.RAW_DATA_DIR / "baf_base_dataset.csv"

paysim_df = load_dataset(paysim_path)
baf_df = load_dataset(baf_path)

# Function to display structural overview
def print_dataset_overview(df: pd.DataFrame, dataset_name: str, target_col: str):
    print(f"==================================================")
    print(f"          {dataset_name.upper()} DATASET OVERVIEW")
    print(f"==================================================")
    print(f"• Shape: {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"• Target Variable: '{target_col}'")
    print(f"• Memory Usage: {df.memory_usage(deep=True).sum() / (1024*1024):.2f} MB")
    print("\n--- Feature Names & Data Types ---")
    dtype_df = pd.DataFrame({
        'Data Type': df.dtypes,
        'Unique Values': df.nunique()
    })
    display(dtype_df)
    print("\n--- Target Class Distribution ---")
    target_dist = df[target_col].value_counts()
    target_pct = df[target_col].value_counts(normalize=True) * 100
    display(pd.DataFrame({'Count': target_dist, 'Percentage (%)': target_pct.round(3)}))

print_dataset_overview(paysim_df, "PaySim Mobile Money", "isFraud")
print_dataset_overview(baf_df, "Bank Account Fraud (BAF)", "fraud_bool")

# SECTION 2: MISSING VALUE ANALYSIS
Inspection of missing/null values across all attributes.

In [ ]:
def analyze_missing_values(df: pd.DataFrame, dataset_name: str):
    missing_count = df.isnull().sum()
    missing_pct = (missing_count / len(df)) * 100
    missing_table = pd.DataFrame({'Missing Count': missing_count, 'Missing (%)': missing_pct})
    
    print(f"--- Missing Value Analysis: {dataset_name} ---")
    total_missing = missing_count.sum()
    print(f"Total Missing Elements: {total_missing}")
    display(missing_table[missing_table['Missing Count'] > 0] if total_missing > 0 else "No Missing Values Found.")
    
    # Matplotlib Missing Value Matrix
    plt.figure(figsize=(10, 3))
    plt.imshow(df.isnull(), aspect='auto', cmap='binary', interpolation='none')
    plt.title(f"Missing Value Heatmap ({dataset_name}) - White indicates Nulls")
    plt.xlabel("Features")
    plt.ylabel("Row Index")
    plt.show()

analyze_missing_values(paysim_df, "PaySim")
analyze_missing_values(baf_df, "Bank Account Fraud")

# SECTION 3: DUPLICATE ANALYSIS
Check for exact duplicate records.

In [ ]:
def analyze_duplicates(df: pd.DataFrame, dataset_name: str):
    num_dups = df.duplicated().sum()
    dup_pct = (num_dups / len(df)) * 100
    print(f"--- Duplicate Analysis: {dataset_name} ---")
    print(f"• Duplicate Records Count: {num_dups}")
    print(f"• Duplicate Percentage: {dup_pct:.3f}%")
    if num_dups > 0:
        print("Sample Duplicate Rows:")
        display(df[df.duplicated(keep=False)].head())
    else:
        print("Confirmation: Zero duplicate rows detected.")
    print("\n")

analyze_duplicates(paysim_df, "PaySim")
analyze_duplicates(baf_df, "Bank Account Fraud")

# SECTION 4: STATISTICAL SUMMARY
Descriptive statistics including mean, median (50%), std, min, max, and quartiles.

In [ ]:
print("--- PaySim Numerical Features Statistical Summary ---")
display(paysim_df.describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']])

print("\n--- Bank Account Fraud (BAF) Numerical Features Statistical Summary ---")
display(baf_df.describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']])

# SECTION 5: TARGET ANALYSIS
Visualizing target class distribution (Fraud vs Legitimate).

In [ ]:
def plot_target_distribution(df: pd.DataFrame, target_col: str, title: str):
    counts = df[target_col].value_counts()
    labels = ['Legitimate (0)', 'Scam/Fraud (1)']
    
    fig = make_subplots(rows=1, cols=2, specs=[[{'type':'bar'}, {'type':'pie'}]],
                        subplot_titles=(f'{title} Target Count', f'{title} Target Percentage'))
    
    fig.add_trace(go.Bar(x=labels, y=counts.values, marker_color=['#28a745', '#dc3545'],
                         text=counts.values, textposition='auto'), row=1, col=1)
    
    fig.add_trace(go.Pie(labels=labels, values=counts.values, hole=0.4,
                         marker_colors=['#28a745', '#dc3545']), row=1, col=2)
    
    fig.update_layout(title_text=f"Target Class Distribution: {title}", showlegend=False, height=400)
    fig.show()

plot_target_distribution(paysim_df, "isFraud", "PaySim Dataset")
plot_target_distribution(baf_df, "fraud_bool", "Bank Account Fraud (BAF) Dataset")

# SECTION 6: NUMERICAL FEATURE ANALYSIS
Histograms, KDE plots, Boxplots, and Violin plots for continuous features.

In [ ]:
def plot_numerical_distributions(df: pd.DataFrame, num_cols: list, dataset_name: str):
    for col in num_cols:
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        
        # 1. Histogram & KDE
        df[col].plot(kind='hist', bins=30, ax=axes[0], density=True, color='#2563EB', alpha=0.6)
        df[col].plot(kind='kde', ax=axes[0], color='#1D4ED8', linewidth=2)
        axes[0].set_title(f"{dataset_name}: {col} Histogram & KDE")
        axes[0].set_xlabel(col)
        
        # 2. Boxplot
        axes[1].boxplot(df[col].dropna(), vert=True, patch_artist=True,
                        boxprops=dict(facecolor='#38BDF8', color='#0284C7'))
        axes[1].set_title(f"{dataset_name}: {col} Boxplot")
        axes[1].set_ylabel(col)
        
        # 3. Violinplot
        axes[2].violinplot(df[col].dropna(), showmeans=True, showmedians=True)
        axes[2].set_title(f"{dataset_name}: {col} Violin Plot")
        axes[2].set_ylabel(col)
        
        plt.tight_layout()
        plt.show()

# Select representative numerical features
plot_numerical_distributions(paysim_df, ['amount', 'oldbalanceOrg', 'newbalanceOrig'], "PaySim")
plot_numerical_distributions(baf_df, ['customer_age', 'velocity_6h', 'credit_risk_score'], "BAF")

# SECTION 7: CATEGORICAL FEATURE ANALYSIS
Distribution and fraud breakdown by categorical attributes.

In [ ]:
def analyze_categorical_features(df: pd.DataFrame, cat_col: str, target_col: str, dataset_name: str):
    counts = df[cat_col].value_counts()
    fraud_rates = df.groupby(cat_col)[target_col].mean() * 100
    
    fig = make_subplots(rows=1, cols=2, subplot_titles=(
        f"{dataset_name}: Count by {cat_col}", f"{dataset_name}: Fraud Rate (%) by {cat_col}"
    ))
    
    fig.add_trace(go.Bar(x=counts.index, y=counts.values, marker_color='#6366F1'), row=1, col=1)
    fig.add_trace(go.Bar(x=fraud_rates.index, y=fraud_rates.values, marker_color='#EF4444'), row=1, col=2)
    
    fig.update_layout(height=400, showlegend=False, title_text=f"Categorical Breakdown: {cat_col} ({dataset_name})")
    fig.show()

analyze_categorical_features(paysim_df, "type", "isFraud", "PaySim")
analyze_categorical_features(baf_df, "payment_type", "fraud_bool", "BAF")
analyze_categorical_features(baf_df, "housing_status", "fraud_bool", "BAF")

# SECTION 8: OUTLIER ANALYSIS (IQR METHOD)
Quantifying extreme values using Interquartile Range (IQR) bounds without dropping points.

In [ ]:
def detect_outliers_iqr(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    num_cols = df.select_dtypes(include=[np.number]).columns
    records = []
    
    for col in num_cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        outlier_count = len(outliers)
        outlier_pct = round((outlier_count / len(df)) * 100, 3)
        
        records.append({
            'Feature': col,
            'Q1 (25%)': round(q1, 2),
            'Q3 (75%)': round(q3, 2),
            'IQR': round(iqr, 2),
            'Lower Bound': round(lower_bound, 2),
            'Upper Bound': round(upper_bound, 2),
            'Outlier Count': outlier_count,
            'Outlier (%)': outlier_pct
        })
    
    res_df = pd.DataFrame(records)
    print(f"--- Outlier Analysis (IQR Method): {dataset_name} ---")
    display(res_df)
    return res_df

_ = detect_outliers_iqr(paysim_df, "PaySim")
_ = detect_outliers_iqr(baf_df, "Bank Account Fraud (BAF)")

# SECTION 9: CORRELATION ANALYSIS
Pearson Correlation Matrix and Plotly Heatmaps.

In [ ]:
def plot_correlation_heatmap(df: pd.DataFrame, dataset_name: str):
    num_df = df.select_dtypes(include=[np.number])
    corr = num_df.corr()
    
    fig = px.imshow(
        corr, text_auto=".2f", aspect="auto",
        color_continuous_scale="RdBu_r",
        title=f"Pearson Correlation Heatmap: {dataset_name}"
    )
    fig.update_layout(height=500)
    fig.show()

plot_correlation_heatmap(paysim_df, "PaySim Mobile Money")
plot_correlation_heatmap(baf_df, "Bank Account Fraud (BAF)")

# SECTION 10: FEATURE VS TARGET RELATIONSHIPS
Analyzing feature differences between Fraudulent and Legitimate transactions.

In [ ]:
# 1. PaySim: Transaction Amount vs Fraud
fig_ps = px.box(
    paysim_df,
    x="isFraud",
    y="amount",
    color="isFraud",
    log_y=True,
    title="PaySim: Transaction Amount Distribution by Fraud Class (Log Scale)",
    labels={"isFraud": "Fraud Flag (0=Legit, 1=Scam)", "amount": "Amount ($)"}
)
fig_ps.show()

# 2. BAF: Velocity 6h vs Fraud
fig_baf = px.box(
    baf_df,
    x="fraud_bool",
    y="velocity_6h",
    color="fraud_bool",
    title="BAF: Application Velocity (Past 6 Hours) by Fraud Class",
    labels={"fraud_bool": "Fraud Flag (0=Legit, 1=Scam)", "velocity_6h": "Velocity 6h"}
)
fig_baf.show()

# SECTION 11: CLASS IMBALANCE EVALUATION
Detailed assessment of target class skewness and its impact on machine learning.

In [ ]:
def evaluate_class_imbalance(df: pd.DataFrame, target_col: str, dataset_name: str):
    counts = df[target_col].value_counts()
    legit = counts.get(0, 0)
    scam = counts.get(1, 0)
    total = len(df)
    
    scam_pct = (scam / total) * 100
    legit_pct = (legit / total) * 100
    imbalance_ratio = round(legit / scam, 2) if scam > 0 else 0
    
    print(f"==================================================")
    print(f"     CLASS IMBALANCE REPORT: {dataset_name.upper()}")
    print(f"==================================================")
    print(f"• Legitimate Count (0): {legit:,} ({legit_pct:.3f}%)")
    print(f"• Fraud/Scam Count (1): {scam:,} ({scam_pct:.3f}%)")
    print(f"• Imbalance Ratio: 1 Scam per {imbalance_ratio} Legitimate transactions")
    print(f"• Machine Learning Impact:")
    print(f"  - A naive baseline classifier predicting 0 for all instances achieves {legit_pct:.2f}% Accuracy.")
    print(f"  - Accuracy is a misleading evaluation metric.")
    print(f"  - Models require PR-AUC, ROC-AUC, Recall, Precision, and class weighting / resampling.\n")

evaluate_class_imbalance(paysim_df, "isFraud", "PaySim")
evaluate_class_imbalance(baf_df, "fraud_bool", "Bank Account Fraud")

# SECTION 12: DATASET COMPARISON
Side-by-side comparison table between PaySim Mobile Money and Bank Account Fraud (BAF).

In [ ]:
comp_table = pd.DataFrame({
    'Comparison Metric': [
        'Domain Context',
        'Total Rows',
        'Total Columns',
        'Numeric Features',
        'Categorical Features',
        'Target Variable',
        'Fraud Count',
        'Fraud Rate (%)',
        'Imbalance Ratio',
        'Missing Values Count',
        'Primary Risk Vectors',
        'Dataset Complexity',
        'Recommended Usage'
    ],
    'PaySim Mobile Money': [
        'Mobile Money Transfer & Cash-out',
        '25,000',
        '11',
        '8',
        '3 (type, nameOrig, nameDest)',
        'isFraud',
        '302',
        '1.208%',
        '1:81',
        '0',
        'Balance wipeout, large transfer amounts',
        'Moderate (High transactional balance signal)',
        'Primary dataset for transaction scam inference'
    ],
    'Bank Account Fraud (BAF)': [
        'Bank Account Opening & Applications',
        '20,000',
        '21',
        '17',
        '4 (payment_type, employment_status, etc.)',
        'fraud_bool',
        '2,557',
        '12.785%',
        '1:7',
        '0',
        'High velocity 6h, device fraud count, credit risk',
        'High (Tabular behavioral attributes)',
        'Secondary validation for identity & application fraud'
    ]
})

display(comp_table)

# SECTION 13: FINAL OBSERVATIONS & TECHNICAL SUMMARY

### 1. Key Data Quality Findings
- **Data Integrity**: Both `PaySim` and `Bank Account Fraud (BAF)` raw CSV files are completely clean of missing values and duplicate rows.
- **Data Types**: Numerical features are properly typed (`float64`, `int64`), and string variables (`type`, `payment_type`) will require one-hot encoding.

### 2. Strong Predictive Feature Candidates
- **PaySim**: 
  - `amount`: Extremely strong correlation with scam transfers.
  - `oldbalanceOrg` vs `newbalanceOrig`: Origin account balance wipeouts (`newbalanceOrig == 0` when `oldbalanceOrg > 0`) strongly signal scam-guided coerced transfers.
  - `type`: Fraud occurs exclusively in `TRANSFER` and `CASH_OUT` transaction types.
- **BAF**:
  - `velocity_6h`: Application frequency in 6-hour windows shows clear positive shift for fraudulent application attempts.
  - `device_fraud_count` & `date_of_birth_distinct_emails_4w`: Account takeover / identity fraud indicators.

### 3. Weak / Redundant Features
- **PaySim**: `nameOrig` and `nameDest` contain high-cardinality nominal string IDs that add noise if not converted to historical frequency or removed.
- **BAF**: `month` shows uniform distribution and minimal correlation with target `fraud_bool`.

### 4. Feature Engineering Concepts (To Be Implemented in Phase 3)
- `balance_error_orig`: Calculated as `abs(oldbalanceOrg - amount - newbalanceOrig)`.
- `balance_error_dest`: Calculated as `abs(oldbalanceDest + amount - newbalanceDest)`.
- `amount_to_balance_ratio`: Transaction amount relative to account balance.

### 5. Preprocessing & Modeling Challenges
- **Extreme Class Imbalance**: PaySim has a 1.208% fraud rate (1:81 ratio). Standard classifiers will suffer from high False Negative rates if unhandled.
- **Outlier Magnitudes**: High variance in transaction amounts requires robust scaling (`StandardScaler` or `RobustScaler`).

> **Status:** Exploratory Data Analysis (EDA) Phase Complete. Ready for Feature Engineering & Preprocessing Phase.